# Big Data Bowl 2027 — Offensive Line Injury Risk from Tracking Data

**Argument:** tracking data can show injury-risk tendencies in offensive linemen that timed Combine drills don't measure, so the draft process would benefit from a tracked, repeated OL drill.

**Plan** (same structure as the DB notebook, `bdb2027_db_injury_risk.ipynb`)
1. **Identify injured OL** — injury defined through snaps (change point on offensive snap share), cross-checked against injury reports.
2. **Tracking tendencies** — pre-injury trajectories vs. matched controls (2a), in-game taper (2b), change points (2c), the **load → response signature** (2d), and season-long player profiles (2e).
3. **Correlate to the Combine** — results (3) and movement inside the tracked drills (3b).
4. **Propose a Combine drill** for OL.

**What's different for OL.** Linemen rarely run fast or make hard cuts, so the DB measures (top speed, turn rate, cuts) mostly don't apply. This notebook measures what OL actually do in the first 1–3 seconds:

| Measure | What it captures | Better is |
|---|---|---|
| `get_off` | Seconds from the snap to 1 yard of movement (interpolated between frames) | lower |
| `first_step` | Peak acceleration in the first 0.8 s | higher |
| `set_depth` | Yards gained backward in the first 1.5 s on pass plays (kick-step) | context |
| `slide_speed` | Peak lateral speed in the first 2.5 s on pass plays (mirroring a rusher) | higher |
| `stop_power` | Peak deceleration in the first 3 s ("stopping power": absorbing contact, redirecting) | higher |
| `effort_per_rep` | Average peak acceleration per rep — how hard he worked each snap (a per-rep rate, like the DB `cuts_per_play`, so partial tracking doesn't distort it) | load |
| `effort_load` | Sum of peak acceleration across a game's tracked reps — volume and intensity combined | load |
| `hi_reps` | Reps whose peak acceleration is in the top 25% of all OL reps | load |
| `pressure_rate` | Pressures allowed per pass-pro rep (from `player_play`) — a performance outcome | lower |

The **signature test (2d)** takes the DB finding (heavy-load game, then a drop in the key movement measure, then injury) and checks it on OL with thresholds fixed in advance. Since it wasn't designed from OL data, it's a genuine replication test.

**Data:** reads `../data/` (run from `Ben_EDA/`). If that folder isn't found, it runs on synthetic demo data from `synthetic_ol.py` (same folder).

## 0. Setup & data loading

In [ ]:
from pathlib import Path
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

# ---------------- config ----------------
DATA_DIR = Path("../data")
TRACKING_GLOB = "game_tracking_*.csv"
USE_SYNTHETIC = not (DATA_DIR / "games.csv").exists()

OL_POSITIONS = {"T", "G", "C", "OT", "OG", "OL"}
SNAP_EVENTS = {"ball_snap", "snap_direct", "autoevent_ballsnap"}

# Step 1 — injury definition
MIN_BASELINE_GAMES = 3
BASELINE_MIN_PCT = 0.60
DROP_RATIO = 0.50
MIN_OUT_GAMES = 1
LONG_ABSENCE_GAMES = 4

# Step 2 — OL rep measures
GETOFF_YDS = 1.0            # get-off = time to this much displacement from the snap spot
GETOFF_MAX_SEC = 1.5        # ignore reps that haven't moved by then (e.g., stand-up / no block)
FIRST_STEP_SEC = 0.8        # window for first-step acceleration
SET_SEC = 1.5               # window for pass-set depth
SLIDE_SEC = 2.5             # window for lateral slide speed
REP_SEC = 3.0               # window for stop power / effort
HI_ACC_PCTL = 0.75          # a rep is "high effort" if its peak accel is in the top 25% of all OL reps
MIN_PLAYS_PER_GAME = 5      # partial tracking: keep games with at least this many reps
PRE_WINDOW = 2
MIN_BASE_GAMES_TRK = 2
N_CONTROLS = 4
CONTROL_SEED = 0

# Step 2d — signature thresholds, fixed in advance from the DB analysis
LOAD_COL = "effort_per_rep"  # load measure (normalized to own baseline); per-rep, like the DB cuts_per_play
RESPONSE_COL = "first_step"  # response measure
LOAD_THR = 1.10              # heavy-load game = load > 1.10x baseline at game -2
DROP_THR = 0.05              # response drop > 5% from game -2 to -1

print("Synthetic demo mode" if USE_SYNTHETIC else f"Real data from {DATA_DIR.resolve()}")

In [ ]:
PP_COLS = ["game_id", "play_id", "nfl_id", "team_abbr", "possession_team", "quarter", "lined_up_position",
           "play_direction", "pass_result", "pressure_allowed", "sack_allowed", "time_to_pressure_allowed"]

def load_repo_data(data_dir, chunksize=5_000_000):
    raw = {k: pd.read_csv(data_dir / f"{k}.csv") for k in ["games", "players", "combine_results"]}
    raw["player_play"] = pd.read_csv(data_dir / "player_play.csv", usecols=lambda c: c in PP_COLS)
    keep = set(raw["players"].loc[raw["players"].nfl_position.isin(OL_POSITIONS), "nfl_id"])
    cols = ["game_id", "play_id", "nfl_id", "time", "x", "y", "s", "a", "dir", "event"]
    parts = []
    for f in sorted(glob.glob(str(data_dir / TRACKING_GLOB))):
        for chunk in pd.read_csv(f, usecols=cols, chunksize=chunksize):
            parts.append(chunk[chunk.nfl_id.isin(keep)])
        print("loaded", Path(f).name)
    raw["game_tracking"] = pd.concat(parts, ignore_index=True)
    raw["combine_tracking"] = pd.concat(
        c[c.nfl_id.isin(keep)] for c in pd.read_csv(data_dir / "combine_tracking.csv", chunksize=chunksize))
    return raw


def load_nflverse(seasons):
    import nflreadpy as nfl
    return dict(snaps=nfl.load_snap_counts(seasons).to_pandas(),
                injuries=nfl.load_injuries(seasons).to_pandas(),
                id_map=nfl.load_players().to_pandas()[["gsis_id", "nfl_id", "pfr_id", "display_name", "position"]],
                schedule=nfl.load_schedules(seasons).to_pandas())


if USE_SYNTHETIC:
    from synthetic_ol import make_all
    raw = make_all()
else:
    raw = load_repo_data(DATA_DIR)

ids = {"game_id": "gameId", "play_id": "playId", "nfl_id": "nflId"}
games = raw["games"].rename(columns=ids)
print("season_type values:", games.season_type.unique())
games = games[games.season_type.astype(str).str.upper().str.startswith("REG")]
SEASONS = sorted(games.season.unique().tolist())

players = raw["players"].rename(columns=ids | {"nfl_position": "position", "display_name": "displayName"})
player_play = raw["player_play"].rename(columns=ids)
for c in PP_COLS:
    if c not in player_play.columns and c not in ids:
        player_play[c] = np.nan
plays = player_play.drop_duplicates(["gameId", "playId"])[["gameId", "playId", "quarter", "play_direction"]]
combine_res = raw["combine_results"].rename(columns=ids)
combine_trk = raw["combine_tracking"].rename(columns=ids)

nv = {k: raw[k] for k in ["snaps", "injuries", "id_map", "schedule"]} if USE_SYNTHETIC else load_nflverse(SEASONS)
snaps, injuries, schedule = nv["snaps"], nv["injuries"], nv["schedule"]

id_map = nv["id_map"].dropna(subset=["nfl_id"]).copy()
id_map["nflId"] = pd.to_numeric(id_map["nfl_id"], errors="coerce")
id_map = id_map.dropna(subset=["nflId"]).drop_duplicates("nflId")

ol_ids = set(players.loc[players.position.isin(OL_POSITIONS), "nflId"])
print(f"seasons: {SEASONS} | OL: {len(ol_ids)} | positions: {players.loc[players.nflId.isin(ol_ids), 'position'].value_counts().to_dict()}")

In [ ]:
def prep_tracking(trk, keys, snap_events=None):
    """Parse time, order frames, estimate sample rate, and (for games) start each rep at the snap."""
    trk = trk.copy()
    trk["time"] = pd.to_datetime(trk.time, format="mixed")
    trk = trk.sort_values(keys + ["time"])
    if snap_events is not None and "event" in trk.columns:
        snap = trk[trk.event.isin(snap_events)].groupby(keys[:-1]).time.min().rename("snap_time")
        trk = trk.join(snap, on=keys[:-1])
        trk["has_snap"] = trk.snap_time.notna()
        trk = trk[~trk.has_snap | (trk.time >= trk.snap_time)].drop(columns="snap_time")
    trk["frameId"] = trk.groupby(keys).cumcount() + 1
    dt = trk.groupby(keys).time.diff().dt.total_seconds()
    fps = float(round(1 / dt.median()))
    return trk.reset_index(drop=True), fps


tracking = raw["game_tracking"].rename(columns=ids)
print("most common events:", tracking.event.value_counts().head(8).to_dict())
tracking = tracking[tracking.gameId.isin(games.gameId)]
tracking, FPS = prep_tracking(tracking, ["gameId", "playId", "nflId"], SNAP_EVENTS)
del raw["game_tracking"]
print(f"game tracking at {FPS:.0f} Hz | {tracking.has_snap.mean():.0%} of rows in plays with a snap event")
print("OL tracked per play:", tracking.groupby(["gameId", "playId"]).nflId.nunique().describe()[["50%", "max"]].to_dict())

## 1. Identify injured OL

Same definition as the DB notebook, on **offensive** snap share: steady usage, then a sudden drop that stays down, found by change point detection and confirmed against the injury report. Long unconfirmed absences are flagged as likely IR.

In [ ]:
def build_snap_grid(snaps, schedule, positions):
    s = snaps[(snaps.game_type == "REG") & snaps.position.isin(positions)].copy()
    sch = schedule[schedule.game_type == "REG"]
    team_games = pd.concat([sch[["week", "home_team"]].rename(columns={"home_team": "team"}),
                            sch[["week", "away_team"]].rename(columns={"away_team": "team"})])
    weeks = np.sort(sch.week.unique())
    grid = (pd.MultiIndex.from_product([s.pfr_player_id.unique(), weeks], names=["pfr_player_id", "week"])
              .to_frame(index=False)
              .merge(s[["pfr_player_id", "week", "team", "offense_pct", "player", "position"]],
                     how="left", on=["pfr_player_id", "week"]))
    grid = grid.sort_values(["pfr_player_id", "week"])
    for c in ["team", "player", "position"]:
        grid[c] = grid.groupby("pfr_player_id")[c].transform(lambda x: x.ffill().bfill())
    grid = grid.merge(team_games, on=["week", "team"])
    grid["offense_pct"] = grid.offense_pct.fillna(0.0)
    return grid.sort_values(["pfr_player_id", "week"]).reset_index(drop=True)


def find_onset(pct, weeks):
    import ruptures as rpt
    if len(pct) < MIN_BASELINE_GAMES + 1:
        return None
    try:
        bkps = rpt.Pelt(model="l2", min_size=1, jump=1).fit(pct.reshape(-1, 1)).predict(pen=0.15)
    except Exception:
        bkps = [len(pct)]
    candidates = sorted(set(b for b in bkps[:-1]) | set(b - 1 for b in bkps[:-1]))
    for i in candidates:
        if i < MIN_BASELINE_GAMES:
            continue
        base = np.median(pct[i - MIN_BASELINE_GAMES:i])
        if base < BASELINE_MIN_PCT:
            continue
        window = pct[i:i + 1 + MIN_OUT_GAMES]
        if pct[i] < DROP_RATIO * base and window.mean() < DROP_RATIO * base:
            low = pct[i:] < DROP_RATIO * base
            games_low = int(np.argmin(low)) if not low.all() else int(low.sum())
            return int(weeks[i]), float(base), games_low
    return None


LEG = ["knee", "ankle", "foot", "hip", "toe", "acl", "mcl", "achilles", "hamstring", "groin", "calf", "quad",
       "thigh", "shin", "leg"]
UPPER = ["shoulder", "wrist", "elbow", "hand", "finger", "pectoral", "chest", "rib", "arm", "neck"]
TRUNK = ["back", "abdomen", "oblique", "core"]

def region(txt):
    t = str(txt).lower()
    if any(k in t for k in ["concussion", "head"]):
        return "concussion"
    for name, keys in [("leg", LEG), ("upper", UPPER), ("trunk", TRUNK)]:
        if any(k in t for k in keys):
            return name
    return "other"


grid = pd.concat([build_snap_grid(snaps[snaps.season == yr], schedule[schedule.season == yr], OL_POSITIONS)
                    .assign(season=yr) for yr in SEASONS], ignore_index=True)
onsets = []
for (yr, pid), g in grid.groupby(["season", "pfr_player_id"]):
    r = find_onset(g.offense_pct.to_numpy(), g.week.to_numpy())
    if r:
        onsets.append(dict(season=yr, pfr_player_id=pid, player=g.player.iloc[0], position=g.position.iloc[0],
                           team=g.team.iloc[0], onset_week=r[0], baseline_pct=r[1], games_low=r[2]))
onsets = pd.DataFrame(onsets)
onsets = onsets.merge(id_map[["pfr_id", "gsis_id", "nflId"]], left_on="pfr_player_id", right_on="pfr_id", how="left")
rep = injuries[injuries.report_status.isin(["Out", "Doubtful", "Questionable"])]

def match_report(row):
    m = rep[(rep.gsis_id == row.gsis_id) & (rep.season == row.season)
            & rep.week.between(row.onset_week, row.onset_week + 1)]
    return m.report_primary_injury.iloc[0] if len(m) else None

onsets["reported_injury"] = onsets.apply(match_report, axis=1)
onsets["status"] = np.select([onsets.reported_injury.notna(), onsets.games_low >= LONG_ABSENCE_GAMES],
                             ["confirmed", "likely_IR_unconfirmed"], "unconfirmed")
onsets["region"] = np.where(onsets.reported_injury.notna(), onsets.reported_injury.map(region), None)
print(onsets.groupby(["season", "status"]).size().unstack(fill_value=0).to_string())
print(f"onsets missing an nflId link: {onsets.nflId.isna().sum()} of {len(onsets)}")

In [ ]:
show = onsets.sort_values("status").head(6)
fig, axes = plt.subplots(2, 3, figsize=(13, 5), sharey=True)
for ax, r in zip(axes.flat, show.itertuples()):
    g = grid[(grid.pfr_player_id == r.pfr_player_id) & (grid.season == r.season)]
    ax.plot(g.week, g.offense_pct, marker="o")
    ax.axvline(r.onset_week, color="crimson", ls="--")
    ax.set_title(f"{r.player} ({r.position}, {r.season}) — {r.status}\n{r.reported_injury or ''}", fontsize=9)
for ax in axes.flat:
    ax.set_xlabel("week")
axes[0, 0].set_ylabel("offensive snap %")
plt.tight_layout(); plt.show()

In [ ]:
tracked = games.groupby("season").week.agg(["min", "max"])
injured = onsets[(onsets.status == "confirmed") & (onsets.region != "concussion") & onsets.nflId.isin(ol_ids)].copy()
injured = injured[injured.apply(lambda r: r.season in tracked.index and
                                tracked.loc[r.season, "min"] < r.onset_week <= tracked.loc[r.season, "max"] + 1,
                                axis=1)]
injured["nflId"] = injured.nflId.astype(int)
print(f"{len(injured)} confirmed OL injuries with onset inside the tracking window")
print(injured.region.value_counts().to_string())
injured[["season", "player", "position", "onset_week", "reported_injury", "region", "games_low"]]

## 2. Tracking tendencies before injury

Each OL rep is measured from the snap: get-off, first-step acceleration, pass-set depth, lateral slide speed, stop power (peak deceleration), and whether it was a high-effort rep. Pass-pro reps also carry pressures allowed. Everything is normalized to the player's own baseline and compared with matched healthy OL.

In [ ]:
REP_KEYS = ["gameId", "playId", "nflId"]

def ol_rep_features(trk, plays, fps):
    """Per-rep OL measures from the snap. Needs play_direction to tell forward from backward."""
    trk = trk[trk.has_snap].sort_values(REP_KEYS + ["frameId"]).merge(plays[["gameId", "playId", "play_direction"]],
                                                                      on=["gameId", "playId"], how="left")
    g = trk.groupby(REP_KEYS, sort=False)
    trk["t"] = (trk.frameId - 1) / fps
    x0, y0 = g.x.transform("first"), g.y.transform("first")
    sign = np.where(trk.play_direction.astype(str).str.lower() == "left", -1, 1)
    trk["fwd"] = (trk.x - x0) * sign                      # + = toward the defense
    trk["disp"] = np.hypot(trk.x - x0, trk.y - y0)
    trk["long_acc"] = g.s.diff() * fps
    trk["long_acc"] = g.long_acc.transform(lambda v: v.rolling(3, center=True, min_periods=1).mean())
    trk["long_acc"] = trk.long_acc.clip(-12, 12)          # beyond ±12 yd/s^2 is a tracking glitch for OL
    trk["lat_speed"] = (g.y.diff().abs() * fps)
    trk["lat_speed"] = g.lat_speed.transform(lambda v: v.rolling(3, center=True, min_periods=1).mean())

    w = lambda sec: trk.t <= sec
    # get-off: interpolate the moment displacement crosses GETOFF_YDS (10 Hz frames are 0.1 s apart,
    # too coarse for get-off differences of a few hundredths)
    d_prev, t_prev = g.disp.shift(), g.t.shift()
    cross = (trk.disp >= GETOFF_YDS) & (d_prev < GETOFF_YDS) & w(GETOFF_MAX_SEC)
    frac = (GETOFF_YDS - d_prev) / (trk.disp - d_prev)
    trk["go_t"] = np.where(cross, t_prev + frac * (trk.t - t_prev), np.nan)
    trk["fs_acc"] = np.where(w(FIRST_STEP_SEC), trk.long_acc, np.nan)
    trk["depth"] = np.where(w(SET_SEC), -trk.fwd, np.nan)
    trk["lat"] = np.where(w(SLIDE_SEC), trk.lat_speed, np.nan)
    trk["acc_w"] = np.where(w(REP_SEC), trk.long_acc, np.nan)
    trk["s_w"] = np.where(w(REP_SEC), trk.s, np.nan)
    rep = (trk.groupby(REP_KEYS)
              .agg(get_off=("go_t", "min"), first_step=("fs_acc", "max"), set_depth=("depth", "max"),
                   slide_speed=("lat", "max"), peak_acc=("acc_w", "max"), stop_power=("acc_w", "min"),
                   top_speed=("s_w", "max"))
              .reset_index())
    rep["stop_power"] = -rep.stop_power
    return rep


rep = ol_rep_features(tracking, plays, FPS)
rep = rep.merge(player_play[["gameId", "playId", "nflId", "pass_result", "pressure_allowed"]],
                on=REP_KEYS, how="left").merge(plays[["gameId", "playId", "quarter"]], on=["gameId", "playId"], how="left")
rep["is_pass"] = rep.pass_result.notna() & (rep.pass_result.astype(str).str.strip() != "")
pa = rep.pressure_allowed.astype(str).str.strip().str.lower()
rep["pressure_allowed"] = pd.to_numeric(pa.replace({"true": 1, "false": 0, "nan": np.nan, "none": np.nan, "": np.nan}), errors="coerce")
HI_ACC = rep.peak_acc.quantile(HI_ACC_PCTL)
rep["hi_rep"] = rep.peak_acc >= HI_ACC
print(f"high-effort threshold: peak accel >= {HI_ACC:.2f} yd/s^2")
rep.loc[~rep.is_pass, ["set_depth", "slide_speed"]] = np.nan          # pass-set measures only on pass plays
print(f"{len(rep):,} OL reps | pass share {rep.is_pass.mean():.0%} | high-effort share {rep.hi_rep.mean():.0%}")
rep[["get_off", "first_step", "set_depth", "slide_speed", "stop_power", "top_speed", "peak_acc"]].describe().T.round(2)

In [ ]:
FEATURES = ["get_off", "first_step", "set_depth", "slide_speed", "stop_power", "top_speed"]
LOAD = ["n_plays", "pass_reps", "effort_per_rep", "effort_load"]

def game_level(rep, by=("nflId", "gameId")):
    q75 = lambda x: x.quantile(0.75)
    q90 = lambda x: x.quantile(0.9)
    return (rep.groupby(list(by))
               .agg(get_off=("get_off", "median"), first_step=("first_step", q75), set_depth=("set_depth", "median"),
                    slide_speed=("slide_speed", q90), stop_power=("stop_power", q90), top_speed=("top_speed", q90),
                    n_plays=("playId", "nunique"), pass_reps=("is_pass", "sum"), hi_reps=("hi_rep", "sum"),
                    effort_load=("peak_acc", "sum"), effort_per_rep=("peak_acc", "mean"),
                    pressures=("pressure_allowed", "sum"))
               .reset_index()
               .assign(pressure_rate=lambda d: d.pressures / d.pass_reps.replace(0, np.nan)))


gl = game_level(rep).merge(games[["gameId", "season", "week"]], on="gameId")
gl_all = gl.copy()
for k in [3, 5, 10, 20]:
    print(f"n_plays >= {k}: {(gl_all.n_plays >= k).mean():.0%} of OL-games kept")
gl = gl[gl.n_plays >= MIN_PLAYS_PER_GAME]
gl[FEATURES + LOAD + ["pressure_rate"]].describe().T.round(3)

### 2a. Matched controls and pre-injury trajectories

Controls: same position group (tackle vs. interior), similar snap share, same season and weeks. `get_off` and `pressure_rate` are "lower is better", so a *rise* is the bad direction for those.

In [ ]:
def pos_group(p):
    return "T" if str(p) in {"T", "OT"} else "IOL"

base_usage = (grid.merge(id_map[["pfr_id", "nflId"]], left_on="pfr_player_id", right_on="pfr_id")
                  .groupby(["season", "nflId"]).offense_pct.median())
pos_of = players.set_index("nflId").position
ever_onset = {(r.season, int(r.nflId)) for r in onsets.dropna(subset=["nflId"]).itertuples()}
gl_weeks = gl.groupby(["season", "nflId"]).week.apply(np.array).to_dict()

controls = []
rng = np.random.default_rng(CONTROL_SEED)
for r in injured.itertuples():
    pool = [p for p in ol_ids
            if (r.season, p) not in ever_onset
            and pos_group(pos_of.get(p)) == pos_group(r.position)
            and abs(base_usage.get((r.season, p), -1) - r.baseline_pct) <= 0.15
            and (gl_weeks.get((r.season, p), np.array([])) < r.onset_week).sum() >= MIN_BASE_GAMES_TRK + PRE_WINDOW]
    for c in rng.choice(pool, min(N_CONTROLS, len(pool)), replace=False):
        controls.append(dict(nflId=int(c), season=r.season, onset_week=r.onset_week, match_of=r.nflId))
controls = pd.DataFrame(controls, columns=["nflId", "season", "onset_week", "match_of"])
subjects = pd.concat([injured[["nflId", "season", "onset_week"]].assign(group="injured", match_of=injured.nflId),
                      controls.assign(group="control")], ignore_index=True)
print(subjects.group.value_counts().to_string())

In [ ]:
def relative_series(gl, subjects, features, diff_features=()):
    """Each subject's games before (pseudo-)onset, indexed -1, -2, ...; `features` as a ratio to his own
    baseline (1.0 = normal), `diff_features` (rates that can be 0) as a difference from baseline."""
    out = []
    for sid, s in enumerate(subjects.itertuples()):
        g = gl[(gl.nflId == s.nflId) & (gl.season == s.season)].sort_values("week").copy()
        pre = g[g.week < s.onset_week].copy()
        if len(pre) < MIN_BASE_GAMES_TRK + PRE_WINDOW:
            continue
        pre["rel_game"] = -np.arange(len(pre), 0, -1)
        base = pre.iloc[:-PRE_WINDOW][list(features) + list(diff_features)].median()
        both = pd.concat([pre, g[g.week == s.onset_week].assign(rel_game=0)])
        both[list(features)] = both[list(features)] / base[list(features)]
        both[list(diff_features)] = both[list(diff_features)] - base[list(diff_features)]
        both["subject"], both["group"], both["match_of"], both["season"] = sid, s.group, s.match_of, s.season
        out.append(both)
    cols = ["rel_game", "subject", "group", "match_of", "season"] + list(features) + list(diff_features)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=cols)


def cliffs_delta(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return (np.sign(a[:, None] - b[None, :])).mean() if len(a) and len(b) else np.nan


def mwu(a, b):
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna()
    b = pd.to_numeric(pd.Series(b), errors="coerce").dropna()
    return stats.mannwhitneyu(a, b).pvalue if len(a) > 1 and len(b) > 1 else np.nan


def compare_pre_window(rel, features):
    pre = (rel[rel.rel_game.between(-PRE_WINDOW, -1)]
             .groupby(["subject", "group"])[features].mean().reset_index())
    rows = []
    for f in features:
        a = pre.loc[pre.group == "injured", f].dropna(); b = pre.loc[pre.group == "control", f].dropna()
        rows.append(dict(feature=f, injured_mean=a.mean(), control_mean=b.mean(), n_injured=len(a),
                         n_control=len(b), cliffs_delta=cliffs_delta(a, b), p_value=mwu(a, b)))
    return pd.DataFrame(rows), pre


DIFF = ["pressure_rate", "hi_reps"]          # counts/rates that can be 0: compared as differences
rel = relative_series(gl, subjects, FEATURES + LOAD, DIFF)
pre_cmp, _ = compare_pre_window(rel, FEATURES + LOAD + DIFF)
print("ratio to own baseline (pressure_rate: difference from baseline)")
pre_cmp.round(3)

In [ ]:
def plot_trajectories(rel, features, title, ncol=5):
    nrow = int(np.ceil(len(features) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.2 * nrow), squeeze=False)
    for ax, f in zip(axes.flat, features):
        sns.lineplot(data=rel[rel.rel_game >= -6], x="rel_game", y=f, hue="group", errorbar=("ci", 90), marker="o",
                     ax=ax, palette={"injured": "crimson", "control": "grey"})
        ax.axhline(0 if f in DIFF else 1, color="k", lw=0.6); ax.axvline(0, color="crimson", ls=":", lw=0.8)
        ax.set_title(f); ax.set_xlabel("games from onset"); ax.set_ylabel("vs. own baseline")
        if ax is not axes.flat[0] and ax.get_legend():
            ax.get_legend().remove()
    for ax in list(axes.flat)[len(features):]:
        ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

plot_trajectories(rel, FEATURES + ["effort_per_rep", "pressure_rate"], "OL pre-injury trajectories (normalized to own baseline)",
                  ncol=4)

### 2b. Within-game taper

Does get-off or first-step acceleration fade from Q1 to Q4, and more so in the games before injury? For OL, a slower get-off late in games is the clearest fatigue sign.

In [ ]:
TAPER_F = ["get_off", "first_step", "stop_power", "slide_speed"]

def taper_slopes(rep, features):
    gq = game_level(rep, by=("nflId", "gameId", "quarter")).merge(games[["gameId", "season", "week"]], on="gameId")
    gq = gq[gq.quarter.between(1, 4) & (gq.n_plays >= 3)]
    rows = []
    for (pid, gid), g in gq.groupby(["nflId", "gameId"]):
        g = g.sort_values("quarter")
        if g.quarter.nunique() < 3 or 1 not in set(g.quarter):
            continue
        q1 = g[g.quarter == 1][features].iloc[0]
        row = dict(nflId=pid, gameId=gid, season=g.season.iloc[0], week=g.week.iloc[0])
        for f in features:
            y = (g[f] / q1[f]).to_numpy(); ok = np.isfinite(y)
            row[f"{f}_taper"] = np.polyfit(g.quarter.to_numpy()[ok], y[ok], 1)[0] if ok.sum() >= 3 else np.nan
        rows.append(row)
    return gq, pd.DataFrame(rows, columns=["nflId", "gameId", "season", "week"] + [f"{f}_taper" for f in features])


gq, taper = taper_slopes(rep, TAPER_F)
TAPER = [f"{f}_taper" for f in TAPER_F]
print(f"{len(taper):,} OL-games with enough plays per quarter for a taper slope")

def subject_weeks(s, table, n=PRE_WINDOW):
    w = table[(table.nflId == s.nflId) & (table.season == s.season) & (table.week < s.onset_week)]
    return w.sort_values("week").tail(n)

taper_rows = []
for sid, s in enumerate(subjects.itertuples()):
    t = subject_weeks(s, taper)
    if len(t):
        taper_rows.append(t[TAPER].mean().to_dict() | dict(subject=sid, group=s.group))
taper_pre = pd.DataFrame(taper_rows, columns=TAPER + ["subject", "group"])
taper_cmp = pd.DataFrame([dict(feature=f,
                               injured_mean=taper_pre.loc[taper_pre.group == "injured", f].mean(),
                               control_mean=taper_pre.loc[taper_pre.group == "control", f].mean(),
                               cliffs_delta=cliffs_delta(taper_pre.loc[taper_pre.group == "injured", f].dropna(),
                                                         taper_pre.loc[taper_pre.group == "control", f].dropna()),
                               p_value=mwu(taper_pre.loc[taper_pre.group == "injured", f].dropna(),
                                           taper_pre.loc[taper_pre.group == "control", f].dropna()))
                          for f in TAPER])
print("Slope per quarter relative to Q1 (get_off: positive = slowing; others: negative = fading)")
taper_cmp.round(4)

### 2c. Change point on the performance series

Does a change point in get-off, first step, or stop power show up before injury, and how often does it fire for healthy controls?

In [ ]:
def perf_changepoint(series, min_change=0.04, worse="down"):
    y = np.asarray(series, float); y = y[np.isfinite(y)]
    if len(y) < 4:
        return None
    best, best_cost = None, np.inf
    for k in range(2, len(y) - 1):
        cost = ((y[:k] - y[:k].mean()) ** 2).sum() + ((y[k:] - y[k:].mean()) ** 2).sum()
        if cost < best_cost:
            best, best_cost = k, cost
    change = y[:best].mean() - y[best:].mean()
    return best if (change if worse == "down" else -change) >= min_change else None


cp_rows = []
for (sid, grp_), g in rel[rel.rel_game < 0].groupby(["subject", "group"]):
    g = g.sort_values("rel_game")
    for f, worse in [("first_step", "down"), ("stop_power", "down"), ("get_off", "up")]:
        k = perf_changepoint(g[f].to_numpy(), worse=worse)
        cp_rows.append(dict(subject=sid, group=grp_, feature=f, fired=k is not None,
                            lead_games=(len(g) - k) if k is not None else np.nan))
cp = pd.DataFrame(cp_rows)
cp.groupby(["feature", "group"]).agg(rate=("fired", "mean"), median_lead_games=("lead_games", "median"),
                                     n=("fired", "size")).unstack("group").round(2)

### 2d. Load → response signature (replication of the DB finding)

DB finding: a heavy-cutting game at game −2, then a turn-rate drop into game −1, about 2x more common before injury. The OL version, with thresholds **fixed in advance**: a high-effort game (`effort_per_rep` > 1.10x baseline) at game −2, then a first-step acceleration drop of more than 5% into game −1. Because the thresholds weren't tuned on OL data, the p-value here is honest.

In [ ]:
def signature_table(rel, load_col=LOAD_COL, resp_col=RESPONSE_COL, load_thr=LOAD_THR, drop_thr=DROP_THR,
                    worse="down"):
    rows = []
    for (sid, grp_), g in rel.groupby(["subject", "group"]):
        s = g.set_index("rel_game")
        if not {-2, -1} <= set(s.index):
            continue
        drop = (s.loc[-2, resp_col] - s.loc[-1, resp_col]) * (1 if worse == "down" else -1)
        rows.append(dict(subject=sid, group=grp_, match_of=g.match_of.iloc[0], season=g.season.iloc[0],
                         load_m2=s.loc[-2, load_col], resp_drop=drop))
    sig = pd.DataFrame(rows)
    sig["signature"] = (sig.load_m2 > load_thr) & (sig.resp_drop > drop_thr)
    return sig


def report_signature(sig, label):
    a, b = sig[sig.group == "injured"], sig[sig.group == "control"]
    t = [[a.signature.sum(), (~a.signature).sum()], [b.signature.sum(), (~b.signature).sum()]]
    orr, p = stats.fisher_exact(t) if len(a) and len(b) else (np.nan, np.nan)
    print(f"{label}: injured {a.signature.mean():.0%} ({a.signature.sum()}/{len(a)}) vs control "
          f"{b.signature.mean():.0%} ({b.signature.sum()}/{len(b)}) | odds ratio {orr:.2f}, Fisher p = {p:.3f}")
    print(f"   load at -2 (median): injured {a.load_m2.median():.2f} vs control {b.load_m2.median():.2f}, "
          f"Cliff's delta {cliffs_delta(a.load_m2.dropna(), b.load_m2.dropna()):.2f}")
    print(f"   response drop -2 -> -1 (median): injured {a.resp_drop.median():.3f} vs control {b.resp_drop.median():.3f}, "
          f"Cliff's delta {cliffs_delta(a.resp_drop.dropna(), b.resp_drop.dropna()):.2f}")


sig = signature_table(rel)
report_signature(sig, f"{LOAD_COL} spike -> {RESPONSE_COL} drop (pre-specified)")
# alternate OL versions (exploratory): same load, other responses
sig_go = signature_table(rel, resp_col="get_off", worse="up")
report_signature(sig_go, f"{LOAD_COL} spike -> get_off slows")
sig_pp = signature_table(rel, load_col="pass_reps", resp_col="first_step")
report_signature(sig_pp, "pass_reps spike -> first_step drop")

In [ ]:
# Robustness for the pre-specified version: thresholds, seasons, injury region
grid_rows = []
for lt in [1.05, 1.10, 1.15, 1.20, 1.30]:
    for dt in [0.03, 0.05, 0.08, 0.10]:
        s = (sig.load_m2 > lt) & (sig.resp_drop > dt)
        ci = s[sig.group == "control"].mean()
        grid_rows.append(dict(load_thr=lt, drop_thr=dt,
                                 ratio=s[sig.group == "injured"].mean() / ci if ci > 0 else (np.inf if s[sig.group == "injured"].any() else np.nan)))
print("injured / control signature rate:")
print(pd.DataFrame(grid_rows).pivot(index="load_thr", columns="drop_thr", values="ratio").round(2).to_string(), "\n")

for yr, g in sig.groupby("season"):
    a, b = g[g.group == "injured"], g[g.group == "control"]
    print(yr, f"injured {a.signature.mean():.0%} (n={len(a)}) vs control {b.signature.mean():.0%} (n={len(b)})")

region_of = {(r.season, r.nflId): r.region for r in injured.itertuples()}
sig["region"] = [region_of.get((s, m)) for s, m in zip(sig.season, sig.match_of)]
print()
print(sig.groupby(["region", "group"]).signature.agg(["mean", "sum", "size"]).round(2).to_string())

### 2e. Player profiles: injured vs. healthy OL seasons

Season-long levels, trends, variability, load, pressure allowed, and Combine results — over each player's games before onset, **excluding the last 2** so the profile isn't just the pre-injury games counted again. Healthy regulars are measured over a matching stretch.

In [ ]:
EXCLUDE_LAST = 2
MIN_GAMES = 4
PROFILE_SEED = 1

tr = gl.sort_values(["nflId", "season", "week"]).copy()
by = tr.groupby(["nflId", "season"])
for f in [LOAD_COL, RESPONSE_COL]:
    tr[f + "_base"] = by[f].transform(lambda x: x.shift(1).rolling(3, min_periods=2).median())
tr["load_norm"] = tr[LOAD_COL] / tr[LOAD_COL + "_base"]
tr["spike"] = tr.load_norm > 1.20
tr["resp_next"] = by[RESPONSE_COL].shift(-1) / tr[RESPONSE_COL + "_base"]
tr["next_week"] = by.week.shift(-1)
tr = tr.merge(taper[["nflId", "gameId"] + TAPER], on=["nflId", "gameId"], how="left")

inj_onset = {(r.season, r.nflId): r.onset_week for r in injured.itertuples()}
rng_p = np.random.default_rng(PROFILE_SEED)
windows = [(s, n, w, "injured") for (s, n), w in inj_onset.items()]
for (s, n), usage in base_usage.items():
    if usage >= BASELINE_MIN_PCT and (s, int(n)) not in ever_onset and n in ol_ids and len(injured):
        windows.append((s, int(n), int(rng_p.choice(injured.onset_week)), "healthy"))

TREND_F = ["get_off", "first_step", "stop_power", "slide_speed", "top_speed"]
rows = []
for s, n, end, grp in windows:
    g = tr[(tr.nflId == n) & (tr.season == s) & (tr.week < end)].sort_values("week")
    if EXCLUDE_LAST:
        g = g.iloc[:-EXCLUDE_LAST]
    if len(g) < MIN_GAMES:
        continue
    x = np.arange(len(g))
    r = dict(season=s, nflId=n, group=grp, games=len(g))
    for f in FEATURES + ["effort_per_rep", "hi_reps", "pass_reps"]:
        r[f"{f}_level"] = g[f].median()
    for f in TREND_F:
        v = g[f] / g[f].median()
        r[f"{f}_variability"] = v.std()
        r[f"{f}_trend"] = np.polyfit(x[v.notna()], v.dropna(), 1)[0] if v.notna().sum() >= MIN_GAMES else np.nan
    r["pressure_rate_level"] = g.pressures.sum() / max(g.pass_reps.sum(), 1)
    r["spike_rate"] = g.spike.mean()
    r["response_after_spike"] = g.loc[g.spike & (g.next_week < end), "resp_next"].median()
    for t in ["get_off_taper", "first_step_taper"]:
        r[t] = g[t].mean()
    rows.append(r)
prof = pd.DataFrame(rows)
prof["pos_group"] = prof.nflId.map(pos_of).map(pos_group)

LEVELS = [c for c in prof.columns if c.endswith("_level")]
prof[LEVELS] = prof.groupby(["season", "pos_group"])[LEVELS].transform(lambda x: (x - x.mean()) / x.std())

COMBINE_COLS = ["forty", "ten_yd_split", "vertical", "broad_jump", "three_cone", "short_shuttle", "bench_reps",
                "arm_length", "hand_size", "combine_weight"]
COMBINE_COLS = [c for c in COMBINE_COLS if c in combine_res.columns]
prof = prof.merge(combine_res.drop_duplicates("nflId", keep="last")[["nflId"] + COMBINE_COLS], on="nflId", how="left")
prof[COMBINE_COLS] = prof.groupby("pos_group")[COMBINE_COLS].transform(lambda x: (x - x.mean()) / x.std())
print(prof.groupby("group").agg(player_seasons=("nflId", "size"), median_games=("games", "median")).to_string())

METRICS = [c for c in prof.columns if c not in ("season", "nflId", "group", "games", "pos_group")]
prof_cmp = []
for m in METRICS:
    a = prof.loc[prof.group == "injured", m].dropna(); b = prof.loc[prof.group == "healthy", m].dropna()
    d = cliffs_delta(a, b)
    by_season = [cliffs_delta(prof.loc[(prof.group == "injured") & (prof.season == s), m].dropna(),
                              prof.loc[(prof.group == "healthy") & (prof.season == s), m].dropna()) for s in SEASONS]
    valid = [v for v in by_season if np.isfinite(v)]
    prof_cmp.append(dict(metric=m, injured_median=a.median(), healthy_median=b.median(), n_injured=len(a),
                         n_healthy=len(b), cliffs_delta=d, p_value=mwu(a, b),
                         same_sign_all_seasons=len(valid) >= 2 and all(np.sign(v) == np.sign(d) for v in valid),
                         season_deltas=[round(v, 2) if np.isfinite(v) else None for v in by_season]))
prof_cmp = pd.DataFrame(prof_cmp).sort_values("cliffs_delta", key=abs, ascending=False).reset_index(drop=True)
display(prof_cmp.round(3))

In [ ]:
top = prof_cmp[(prof_cmp.n_injured >= 8) & (prof_cmp.n_healthy >= 8)].metric.head(8).tolist()
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, m in zip(axes.flat, top):
    sns.boxplot(data=prof, x="group", y=m, hue="group", legend=False, order=["healthy", "injured"], ax=ax,
                palette={"injured": "crimson", "healthy": "lightgrey"}, showfliers=False)
    sns.stripplot(data=prof[prof.group == "injured"], x="group", y=m, order=["healthy", "injured"],
                  ax=ax, color="crimson", size=3, alpha=0.6)
    d = prof_cmp.set_index("metric").loc[m]
    ax.set_title(f"{m}\nδ={d.cliffs_delta:.2f}, p={d.p_value:.2f}", fontsize=9); ax.set_xlabel("")
for ax in list(axes.flat)[len(top):]:
    ax.axis("off")
plt.suptitle("Injured vs. healthy OL seasons: biggest differences (metrics with 8+ players per group)")
plt.tight_layout(); plt.show()

## 3. Correlate to the Combine

For OL, the 10-yard split is the Combine's closest match to get-off and first-step acceleration, and the short shuttle to lateral slide. Do they actually line up with the game measures, and do Combine results differ between injured and healthy OL?

In [ ]:
GAME_COLS = FEATURES + ["get_off_taper", "first_step_taper"]
player_tbl = (gl.groupby("nflId")[FEATURES].median()
                .join(taper.groupby("nflId")[["get_off_taper", "first_step_taper"]].mean())
                .reset_index())
player_tbl["injured"] = player_tbl.nflId.isin(injured.nflId)
player_tbl = player_tbl.merge(combine_res.drop_duplicates("nflId", keep="last")[["nflId"] + COMBINE_COLS],
                              on="nflId", how="left")
print(f"{player_tbl[COMBINE_COLS].notna().any(axis=1).sum()} of {len(player_tbl)} tracked OL have Combine results")

def corr_heatmap(df, rows, cols, title, figsize=(9, 4.5)):
    c = df[rows + cols].corr(method="spearman").loc[rows, cols]
    plt.figure(figsize=figsize)
    sns.heatmap(c, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1)
    plt.title(title); plt.tight_layout(); plt.show()
    return c

player_tbl["pos_group"] = player_tbl.nflId.map(pos_of).map(pos_group)
z = player_tbl.copy()
zcols = COMBINE_COLS + GAME_COLS
z[zcols] = z.groupby("pos_group")[zcols].transform(lambda x: (x - x.mean()) / x.std())
corr = corr_heatmap(z, COMBINE_COLS, GAME_COLS, "Spearman within position group: Combine vs. in-game OL")

In [ ]:
comb_cmp = []
for c in COMBINE_COLS:
    a = player_tbl.loc[player_tbl.injured, c].dropna(); b = player_tbl.loc[~player_tbl.injured, c].dropna()
    comb_cmp.append(dict(drill=c, injured_median=a.median(), healthy_median=b.median(), n_injured=len(a),
                         n_healthy=len(b), cliffs_delta=cliffs_delta(a, b), p_value=mwu(a, b)))
pd.DataFrame(comb_cmp).round(3)

### 3b. Movement inside the Combine drills

From `combine_tracking.csv`, per attempt: first-step acceleration and time to 1 and 10 yards from the start (the tracked version of the get-off and 10-yard split), peak deceleration, and — in the agility drills — turn rate and speed kept through each turn. Then: which tracked measures match the game measures better than the times do?

In [ ]:
print(combine_trk.groupby(["drill_type", "drill_name"]).event_id.nunique().rename("attempts").to_string())

COMBINE_KEYS = ["nflId", "event_id", "drill_name", "attempt"]
MIN_SPEED_FOR_DIR, HARD_CUT_DEG_S, RETENTION_SEC = 2.0, 150, 0.5

def combine_attempt_features(ct, fps):
    ct = ct.sort_values(COMBINE_KEYS + ["frameId"]).copy()
    g = ct.groupby(COMBINE_KEYS, sort=False)
    moving = ct.s >= 0.3                                                   # attempt starts when he moves
    ct["start_t"] = ct.time.where(moving).groupby([ct[k] for k in COMBINE_KEYS]).transform("min")
    ct["t"] = (ct.time - ct.start_t).dt.total_seconds()
    ct = ct[ct.t >= 0]
    g = ct.groupby(COMBINE_KEYS, sort=False)
    x0, y0 = g.x.transform("first"), g.y.transform("first")
    ct["disp"] = np.hypot(ct.x - x0, ct.y - y0)
    ct["long_acc"] = g.s.diff() * fps
    ct["long_acc"] = g.long_acc.transform(lambda v: v.rolling(3, center=True, min_periods=1).mean())
    unwrapped = g.dir.transform(lambda d: np.rad2deg(np.unwrap(np.deg2rad(d.to_numpy()))))
    ct["turn"] = unwrapped.groupby([ct[k] for k in COMBINE_KEYS]).diff().abs() * fps
    ct.loc[ct.s < MIN_SPEED_FOR_DIR, "turn"] = np.nan
    w = max(1, int(round(RETENTION_SEC * fps)))
    is_cut = (ct.turn >= HARD_CUT_DEG_S) & (ct.turn == g.turn.transform(lambda v: v.rolling(5, center=True, min_periods=1).max()))
    s_b, s_a = g.s.shift(w), g.s.shift(-w)
    ct["retention"] = np.where(is_cut & (s_b > MIN_SPEED_FOR_DIR), s_a / s_b, np.nan)
    ct["fs"] = np.where(ct.t <= FIRST_STEP_SEC, ct.long_acc, np.nan)
    ct["t1"] = np.where(ct.disp >= 1, ct.t, np.nan)
    ct["t10"] = np.where(ct.disp >= 10, ct.t, np.nan)
    return (ct.groupby(COMBINE_KEYS)
              .agg(first_step=("fs", "max"), time_1yd=("t1", "min"), time_10yd=("t10", "min"),
                   max_decel=("long_acc", "min"), max_turn=("turn", "max"), retention=("retention", "mean"))
              .assign(max_decel=lambda d: -d.max_decel).reset_index())


ct = combine_trk[combine_trk.nflId.isin(ol_ids)].dropna(subset=["nflId"])
ct, CT_FPS = prep_tracking(ct, COMBINE_KEYS)
print(f"combine tracking at {CT_FPS:.0f} Hz")
attempts = combine_attempt_features(ct, CT_FPS)
ct_player = attempts.groupby(["nflId", "drill_name"])[["first_step", "time_1yd", "time_10yd", "max_decel",
                                                       "max_turn", "retention"]].median().unstack("drill_name")
ct_player.columns = [f"{d}|{f}" for f, d in ct_player.columns]
ct_player = ct_player.dropna(axis=1, thresh=10).reset_index()
CT_COLS = [c for c in ct_player.columns if c != "nflId"]

pt = player_tbl.merge(ct_player, on="nflId", how="left")
drill_vs_game = (pt[CT_COLS + ["get_off", "first_step", "stop_power", "slide_speed"]].corr(method="spearman")
                   .loc[CT_COLS, ["get_off", "first_step", "stop_power", "slide_speed"]])
drill_vs_game["n"] = [pt[[c, "first_step"]].dropna().shape[0] for c in CT_COLS]
drill_vs_game.sort_values("first_step", key=abs, ascending=False).round(2)

In [ ]:
# Fatigue proxy: change from attempt 1 to attempt 2 in each drill, vs. in-game taper
fat = attempts.pivot_table(index=["nflId", "drill_name"], columns="attempt", values="first_step").dropna()
if {1, 2} <= set(fat.columns):
    fat = (fat[2] / fat[1]).rename("attempt2_vs_1").reset_index().pivot(index="nflId", columns="drill_name",
                                                                        values="attempt2_vs_1")
    fat.columns = [f"{c}|first_step_a2/a1" for c in fat.columns]
    fp = player_tbl.merge(fat.reset_index(), on="nflId")
    display(fp[list(fat.columns) + ["first_step_taper", "get_off_taper"]].corr(method="spearman")
              .loc[list(fat.columns), ["first_step_taper", "get_off_taper"]].round(2))

### Summary table

In [ ]:
summary = pd.concat([
    pre_cmp.assign(test="pre-injury level (2a)"),
    taper_cmp.assign(test="in-game taper (2b)"),
])[["test", "feature", "injured_mean", "control_mean", "cliffs_delta", "p_value"]]
summary["abs_effect"] = summary.cliffs_delta.abs()
display(summary.sort_values("abs_effect", ascending=False).round(3))
display(prof_cmp[prof_cmp.same_sign_all_seasons & (prof_cmp.n_injured >= 8)].head(8).round(3))
best = pd.DataFrame({"best |rho|, Combine results": corr.abs().max(axis=0).reindex(["get_off", "first_step", "stop_power", "slide_speed"]),
                     "best |rho|, tracked drills": drill_vs_game.drop(columns="n").abs().max(axis=0)})
display(best.round(2))

## 4. Drill proposal (draft — fill in with results)

**Tracked Repeated Get-Off & Mirror.** From a stance, the lineman reacts to a light or snap cue, explodes out (get-off and first step), sets, and mirrors a cue that changes direction for about 3 seconds, stopping and redirecting on each change. 8 reps with about 20 seconds of rest, tracked by the same chips used in games.

| Drill measure | Game measure it matches | Section |
|---|---|---|
| Reaction to the cue + time to 1 yard | `get_off` (the Combine's self-started 10-yard split can't measure reaction) | 2a, 3b |
| Peak acceleration in the first 0.8 s | `first_step` | 2a, 2d, 3b |
| Peak lateral speed while mirroring | `slide_speed` | 2a |
| Peak deceleration on each direction change | `stop_power` | 2a, 3b |
| **Fade index:** reps 7–8 ÷ reps 1–2 for get-off and first step | taper (2b), load → response signature (2d), trends (2e) | 2b, 2d, 2e |

**Argument:** *[fill in]* The OL measures that separated injured from matched healthy linemen were ___. The best any Combine result correlated with them was ρ = ___ (ρ = ___ for movement tracked inside current drills). Whether the DB load → response signature replicated in OL: ___.

**Caveats:** small number of injured OL; injury from snaps and reports, not medical records; partial tracking per play; get-off depends on the snap event being recorded accurately; pass/run is inferred from `pass_result`.